# RAG v1 exploration: dataset, chunking and retrieval

Scratch space for making the RAG v1 design choices **before** writing `services/rag/`. By the end you should be able to fill in the *Decisions* table at the bottom, which feeds the RAG service brief (Day 2 in the implementation plan):

1. **Chunking strategy** and chunk size
2. **Embedding model** and dimensions
3. **Retrieval mode**: keyword (BM25), semantic (k-NN), or hybrid, plus the hybrid weights
4. **top-k** passed to the agent

**How the notebook is organised**

| Part | What | Needs |
| --- | --- | --- |
| 1 | Explore the knowledge base | nothing |
| 2 | Compare chunking strategies | nothing |
| 3 | A small throwaway eval set + metrics | nothing |
| 4 | Keyword vs semantic vs hybrid, all in memory | Bedrock (Titan embeddings) for 4b onward |
| 5 | Set up OpenSearch and replay the chosen config there | Docker (local) or an AWS domain |
| 6 | Decisions | you |

Parts 1–4 run in memory so you can iterate in seconds without any infrastructure. Part 5 checks that OpenSearch reproduces what you picked.

**Run it:** `uv run --group notebook jupyter lab notebooks/`. In VS Code, pick the `.venv` kernel after running `uv sync --group notebook`.

In [1]:
import hashlib
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import config  # project config (region etc.)

KB_DIR = ROOT / "data" / "knowledge_base"
CACHE_DIR = ROOT / "notebooks" / ".cache"   # embedding cache (gitignored)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
REGION = config.get_aws_region()

pd.set_option("display.max_colwidth", 120)
print(f"repo root: {ROOT}\nregion:    {REGION}")

repo root: /Users/lilianabadillo/Projects/repo/customer-service-assistant
region:    us-east-1


## Part 1. Explore the knowledge base

Look at the size and shape of the documents first. These decide which chunking strategies make sense.

In [2]:
def load_documents(kb_dir: Path) -> list[dict]:
    docs = []
    for path in sorted(kb_dir.rglob("*.md")):
        text = path.read_text(encoding="utf-8")
        title = next((l[2:].strip() for l in text.splitlines() if l.startswith("# ")), path.stem)
        docs.append({
            "doc_id": path.stem,
            "path": str(path.relative_to(ROOT)),
            "category": path.parent.name if path.parent != kb_dir else "general",
            "title": title,
            "text": text,
        })
    return docs


def approx_tokens(text: str) -> int:
    """~4 characters per token for English. Good enough to compare chunk sizes."""
    return round(len(text) / 4)


docs = load_documents(KB_DIR)
DOC_BY_ID = {d["doc_id"]: d for d in docs}

stats = pd.DataFrame([{
    "doc_id": d["doc_id"],
    "category": d["category"],
    "words": len(d["text"].split()),
    "approx_tokens": approx_tokens(d["text"]),
    "sections": sum(1 for l in d["text"].splitlines() if l.startswith("## ")),
    "qa_pairs": len(re.findall(r"^\*\*.+\?\*\*\s*$", d["text"], flags=re.M)),
} for d in docs]).sort_values("words", ascending=False)

print(f"{len(docs)} documents, {stats.words.sum()} words, ~{stats.approx_tokens.sum()} tokens in total")
stats

11 documents, 3045 words, ~4680 tokens in total


,doc_id,category,words,approx_tokens,sections,qa_pairs
0,aws-developers-discount,general,658,1018,6,7
2,hair-type-guide,general,327,494,7,0
1,faq,general,283,452,5,12
4,define-curl-cream,products,235,354,6,3
7,repair-bond-oil,products,231,350,6,3
10,weekly-repair-mask,products,226,336,6,3
8,scalp-reset-serum,products,223,338,6,3
9,silk-slip-conditioner,products,221,354,6,3
6,hydra-wash-shampoo,products,217,336,6,3
3,air-dry-styling-gel,products,216,323,6,3


In [3]:
# Which section headings repeat across documents? (shared templates = structure you can chunk on)
section_counts = Counter(
    l[3:].strip() for d in docs for l in d["text"].splitlines() if l.startswith("## ")
)
pd.Series(section_counts, name="docs").sort_values(ascending=False).to_frame()

,docs
What it is,9
Common questions,9
How to use,8
Key ingredients,8
Who it's for,8
Ingredients & safety,8
About our products,1
Shipping,1
Returns & refunds,1
Subscriptions,1


In [4]:
STOPWORDS = set("""a an and are as at be but by can do does for from how i if in is it its my
of on or so that the their this to was what when which will with you your yes no not""".split())


def tokenize(text: str) -> list[str]:
    """Lowercase word tokens without stopwords. No stemming, so 'iron' and 'irons' differ."""
    return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOPWORDS]


# Document frequency: how many of the docs contain each term.
# Terms found in most docs have low IDF, so BM25 can't use them to tell documents apart.
doc_freq = Counter(t for d in docs for t in set(tokenize(d["text"])))
pd.Series(doc_freq, name="n_docs").sort_values(ascending=False).head(20).to_frame().T

,free,use,hair,cruelty,safe,treated,s,color,vegan,ingredients,common,who,questions,dry,price,5,fl,reviews,4,safety
n_docs,11,11,11,10,10,10,10,10,10,9,9,9,9,9,9,9,8,8,8,8


**What to notice in this corpus**

- **It is tiny.** About 3k words in total, and the longest doc is under 1k tokens. Every doc fits in one Titan embedding call (8k-token limit), so *no chunking* (one chunk per doc) is a real baseline, not a strawman. The cost of whole-doc chunks shows up later: each retrieved passage puts many more tokens into the LLM prompt.
- **Product pages share one template:** *What it is / Who it's for / Key ingredients / How to use / Ingredients & safety / Common questions*. Structure-aware chunking fits this well.
- **Sections don't name their product.** A "How to use" chunk cut from its page doesn't say *which* product it belongs to. Watch for this in Part 2: adding the doc title to each chunk ("contextual headers") fixes it.
- **Many terms repeat across docs:** *color-safe, vegan, cruelty-free, hair* appear almost everywhere. Keyword search can't tell products apart on those words.
- **Q&A-shaped content:** every product page, the FAQ and the discount doc contain `**Question?**` + answer pairs that read like real user questions. One chunk per Q&A pair is worth testing.
- **The discount doc contains instructions for the assistant** ("Do not offer the discount... unless..."). That is agent policy stored in the knowledge base. Note it for Day 5: decide whether such rules belong in the agent's system prompt rather than in retrieved text.

## Part 2. Chunking options

| Strategy | How | Pros | Cons |
| --- | --- | --- | --- |
| `whole_doc` | 1 chunk = 1 file | Full context, trivial | Long passages: more prompt tokens, diluted embeddings |
| `fixed_*w` | N words with overlap | Simple, predictable size (what most managed services default to) | Cuts mid-section, mixes topics |
| `recursive_500c` | Splits on `## `, blank line, line, sentence until ≤ N chars | Respects boundaries when it can | Size-driven, not meaning-driven |
| `markdown` | 1 chunk per `##` section | Matches the doc template | Chunks lose the product name |
| `markdown_ctx` | `markdown` + `"<doc title> \| <section>"` prefix | Each chunk names its product | Prefix repeats in every chunk |
| `qa_aware` | `markdown_ctx`, but Q&A sections split into one chunk per question | Chunks line up with user questions | Many small chunks; answers lose surrounding context |

For the v1/v2 comparison on Day 3, note that Bedrock Knowledge Bases offers *fixed-size* (token-based, with overlap), *hierarchical*, *semantic* and *no chunking*, plus custom chunking through a Lambda. Whichever v1 strategy you pick here, record how it differs from v2's.

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter


def make_chunk(doc: dict, i: int, text: str, section: str = "") -> dict:
    return {
        "chunk_id": f"{doc['doc_id']}#{i}",
        "doc_id": doc["doc_id"],
        "title": doc["title"],
        "section": section,
        "text": text.strip(),
    }


def chunk_whole(doc):
    return [make_chunk(doc, 0, doc["text"])]


def chunk_fixed(doc, size_words=100, overlap_words=20):
    words = doc["text"].split()
    step = size_words - overlap_words
    starts = range(0, max(len(words) - overlap_words, 1), step)
    return [make_chunk(doc, i, " ".join(words[s:s + size_words])) for i, s in enumerate(starts)]


def chunk_recursive(doc, chunk_size=500, chunk_overlap=75):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n\n", "\n", ". ", " "],
    )
    return [make_chunk(doc, i, t) for i, t in enumerate(splitter.split_text(doc["text"]))]


_md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "title"), ("##", "section")], strip_headers=False
)


def chunk_markdown(doc, add_context=False):
    chunks = []
    for i, part in enumerate(_md_splitter.split_text(doc["text"])):
        section = part.metadata.get("section", "")
        text = part.page_content
        if add_context and section:  # the intro part already starts with "# Title"
            text = f"{doc['title']} | {section}\n{text}"
        chunks.append(make_chunk(doc, i, text, section))
    return chunks


QA_PATTERN = re.compile(r"^\*\*(.+\?)\*\*\s*$", re.M)


def chunk_qa_aware(doc):
    pieces = []
    for part in _md_splitter.split_text(doc["text"]):
        section = part.metadata.get("section", "")
        body = part.page_content
        prefix = f"{doc['title']} | {section}\n" if section else ""
        qs = list(QA_PATTERN.finditer(body))
        if len(qs) >= 2:  # a Q&A list: one chunk per question + answer
            for j, m in enumerate(qs):
                end = qs[j + 1].start() if j + 1 < len(qs) else len(body)
                pieces.append((section, prefix + body[m.start():end].strip()))
        else:
            pieces.append((section, prefix + body))
    return [make_chunk(doc, i, t, s) for i, (s, t) in enumerate(pieces)]


STRATEGIES = {
    "whole_doc": chunk_whole,
    "fixed_100w": lambda d: chunk_fixed(d, 100, 20),
    "fixed_50w": lambda d: chunk_fixed(d, 50, 10),
    "recursive_500c": chunk_recursive,
    "markdown": chunk_markdown,
    "markdown_ctx": lambda d: chunk_markdown(d, add_context=True),
    "qa_aware": chunk_qa_aware,
}

chunk_sets = {name: [c for d in docs for c in fn(d)] for name, fn in STRATEGIES.items()}

pd.DataFrame({
    name: {
        "chunks": len(cs),
        "mean_tokens": round(np.mean([approx_tokens(c["text"]) for c in cs])),
        "min_tokens": min(approx_tokens(c["text"]) for c in cs),
        "max_tokens": max(approx_tokens(c["text"]) for c in cs),
    }
    for name, cs in chunk_sets.items()
}).T

,chunks,mean_tokens,min_tokens,max_tokens
whole_doc,11,425,323,1018
fixed_100w,40,138,67,168
fixed_50w,79,72,23,86
recursive_500c,51,91,9,123
markdown,76,61,19,326
markdown_ctx,76,70,28,345
qa_aware,105,54,28,222


In [6]:
def peek(strategy: str, doc_id: str, n: int = 4):
    """Print the first n chunks a strategy produced for one document."""
    for c in [c for c in chunk_sets[strategy] if c["doc_id"] == doc_id][:n]:
        print(f"--- {c['chunk_id']}  (~{approx_tokens(c['text'])} tokens)\n{c['text']}\n")


peek("fixed_50w", "repair-bond-oil")

--- repair-bond-oil#0  (~76 tokens)
# Repair Bond Oil **Category:** Treatment — Repair Oil **Price:** $32 (3.4 fl oz) **Rating:** 4.8 / 5 (1,210 reviews) ## What it is A strengthening treatment oil that helps rebuild and protect hair weakened by heat, bleach, and chemical processing. It smooths the surface for shine while supporting the

--- repair-bond-oil#1  (~84 tokens)
processing. It smooths the surface for shine while supporting the hair's internal bonds. ## Who it's for Best for damaged, color-treated, and chemically processed hair (bleached, relaxed, or permed). Great for anyone fighting breakage and split ends. ## Key ingredients - Bond-building Complex — helps reconnect broken bonds to reduce

--- repair-bond-oil#2  (~70 tokens)
- Bond-building Complex — helps reconnect broken bonds to reduce breakage. - Argan Oil — adds slip, softness, and shine. - Vitamin E — an antioxidant that conditions and protects. ## How to use Apply 2–3 drops to damp mid-lengths and ends before stylin

In [7]:
# Compare: same document, structure-aware with context headers
peek("markdown_ctx", "repair-bond-oil")

--- repair-bond-oil#0  (~30 tokens)
# Repair Bond Oil  
**Category:** Treatment — Repair Oil
**Price:** $32 (3.4 fl oz)
**Rating:** 4.8 / 5 (1,210 reviews)

--- repair-bond-oil#1  (~59 tokens)
Repair Bond Oil | What it is
## What it is
A strengthening treatment oil that helps rebuild and protect hair weakened by heat, bleach, and chemical processing. It smooths the surface for shine while supporting the hair's internal bonds.

--- repair-bond-oil#2  (~48 tokens)
Repair Bond Oil | Who it's for
## Who it's for
Best for damaged, color-treated, and chemically processed hair (bleached, relaxed, or permed). Great for anyone fighting breakage and split ends.

--- repair-bond-oil#3  (~58 tokens)
Repair Bond Oil | Key ingredients
## Key ingredients
- Bond-building Complex — helps reconnect broken bonds to reduce breakage.
- Argan Oil — adds slip, softness, and shine.
- Vitamin E — an antioxidant that conditions and protects.



In [8]:
peek("qa_aware", "faq", n=6)

--- faq#0  (~50 tokens)
AnyCompany Hair Care — General FAQ | About our products
**Are AnyCompany products vegan and cruelty-free?**
Yes. Every AnyCompany product is 100% vegan and cruelty-free, and never tested on animals.

--- faq#1  (~54 tokens)
AnyCompany Hair Care — General FAQ | About our products
**Are your products safe for color-treated hair?**
Yes. The full AnyCompany range is color-safe. Our cleansers are sulfate-free so they won't strip or fade color.

--- faq#2  (~54 tokens)
AnyCompany Hair Care — General FAQ | About our products
**What do you leave out of your formulas?**
We formulate without sulfates, parabens, and (in most products) silicones. Individual product pages list specifics.

--- faq#3  (~48 tokens)
AnyCompany Hair Care — General FAQ | About our products
**Where are AnyCompany products made?**
AnyCompany products are formulated and made in the USA with globally sourced clean ingredients.

--- faq#4  (~41 tokens)
AnyCompany Hair Care — General FAQ | Shipping
**Ho

Things to look for in the chunks above:

- `fixed_50w` starts and ends chunks mid-sentence and mixes "How to use" with "Ingredients & safety". Its embeddings average two topics.
- In `markdown`, the "How to use" chunk doesn't say which product it describes. `markdown_ctx` does.
- `qa_aware` turns FAQ entries into small chunks that read like user questions. Check in Part 4 whether that helps or makes the chunks too thin.

Change sizes, overlaps or prefixes in `STRATEGIES` and rerun. Parts 3–4 pick up whatever is in `chunk_sets`.

## Part 3. A small eval set

Comparing options needs numbers, so here are 18 hand-written questions labelled with the document(s) that answer them. **This is a throwaway seed, not the golden set.** The Day 4 golden set replaces it with expected answers and many more questions, and you can start it from these.

Each question is tagged:
- `lexical`: shares key words with the source doc. BM25 should handle these.
- `paraphrase`: uses different words from the doc ("greasy" vs "oily", "money back" vs "refund"). These need semantic search.

**Metrics** (labels are per document, so a chunk counts as relevant if it comes from a relevant document):
- `hit@1`, `hit@3`: is a relevant doc's chunk at rank 1, or in the top 3?
- `mrr`: mean reciprocal rank of the first relevant chunk, within the top 10
- `ctx_tokens@3`: approximate tokens in the top-3 chunks. This measures how much the LLM has to read, which drives cost and noise.

With only 11 docs and 18 questions, one question = 0.056 of a metric. Use the numbers to **rule out clearly worse options**. Don't treat a 0.05 difference as real.

In [9]:
EVAL_SET = [
    {"q": "Is the bond oil safe for bleached hair?", "relevant": ["repair-bond-oil"], "type": "lexical"},
    {"q": "How much does shipping cost?", "relevant": ["faq"], "type": "lexical"},
    {"q": "Up to what temperature does the heat protectant work?", "relevant": ["heat-shield-mist"], "type": "lexical"},
    {"q": "Is niacinamide in any of your products?", "relevant": ["scalp-reset-serum"], "type": "lexical"},
    {"q": "Does the conditioner contain silicones?", "relevant": ["silk-slip-conditioner"], "type": "lexical"},
    {"q": "Is there a holiday sale?", "relevant": ["aws-developers-discount"], "type": "lexical"},
    {"q": "Are your products tested on animals?", "relevant": ["faq"], "type": "lexical"},
    {"q": "How often should I use the deep conditioning mask?", "relevant": ["weekly-repair-mask"], "type": "lexical"},
    {"q": "What routine do you recommend for fine hair?", "relevant": ["hair-type-guide"], "type": "lexical"},
    {"q": "What should I spray on before using my flat iron?", "relevant": ["heat-shield-mist"], "type": "paraphrase"},
    {"q": "My scalp gets greasy really fast, what can help?", "relevant": ["scalp-reset-serum", "hair-type-guide"], "type": "paraphrase"},
    {"q": "Do you have any promo codes?", "relevant": ["aws-developers-discount"], "type": "paraphrase"},
    {"q": "Which product gives my curls hold without making them stiff?", "relevant": ["air-dry-styling-gel", "define-curl-cream"], "type": "paraphrase"},
    {"q": "Can I get my money back if I already opened the bottle?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "How many days do I have to send something back?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "Which shampoo won't wash out my dye job?", "relevant": ["hydra-wash-shampoo"], "type": "paraphrase"},
    {"q": "When will I see the money from my return on my card?", "relevant": ["faq"], "type": "paraphrase"},
    {"q": "Will the product with argan oil make thin hair look oily?", "relevant": ["repair-bond-oil"], "type": "paraphrase"},
]
assert all(r in DOC_BY_ID for item in EVAL_SET for r in item["relevant"]), "unknown doc_id in EVAL_SET"

METRICS = ["hit@1", "hit@3", "mrr", "ctx_tokens@3"]


def evaluate(search_fn, k: int = 10) -> pd.DataFrame:
    """search_fn(question, k) -> ranked list of chunk dicts (each needs 'doc_id' and 'text')."""
    rows = []
    for item in EVAL_SET:
        hits = search_fn(item["q"], k)
        ranking = [h["doc_id"] for h in hits]
        first = next((i + 1 for i, d in enumerate(ranking) if d in item["relevant"]), None)
        rows.append({
            "q": item["q"], "type": item["type"],
            "hit@1": float(first == 1),
            "hit@3": float(first is not None and first <= 3),
            "mrr": 1 / first if first else 0.0,
            "ctx_tokens@3": sum(approx_tokens(h["text"]) for h in hits[:3]),
            "top3": ranking[:3],
        })
    return pd.DataFrame(rows)


def summarize(results: dict) -> pd.DataFrame:
    return pd.DataFrame({name: df[METRICS].mean() for name, df in results.items()}).T.round(3)

## Part 4. Retrieval: keyword, semantic, hybrid (in memory)

`LocalIndex` holds one chunking strategy and implements the search modes OpenSearch will run in Part 5:

| Mode | Local implementation | OpenSearch equivalent |
| --- | --- | --- |
| keyword | BM25 (`rank_bm25`) | `match` / `multi_match` (BM25 is the default similarity) |
| semantic | cosine similarity over Titan embeddings | `knn` query on a `knn_vector` field |
| hybrid_weighted | min-max normalise both scores, then `alpha * semantic + (1 - alpha) * keyword` | `hybrid` query + `normalization-processor` (min_max, arithmetic_mean, weights) |
| hybrid_rrf | reciprocal rank fusion: `sum 1 / (60 + rank)` | `hybrid` query + `score-ranker-processor` (RRF, OpenSearch 2.19+) |

In [10]:
from rank_bm25 import BM25Okapi


class LocalIndex:
    def __init__(self, chunks: list[dict], dims: int | None = None):
        self.chunks = chunks
        self.dims = dims
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
        self._emb = None  # embeddings are computed the first time they are needed (Part 4b)

    @property
    def emb(self) -> np.ndarray:
        if self._emb is None:
            self._emb = embed_texts([c["text"] for c in self.chunks], self.dims)
        return self._emb

    # raw score vectors, one score per chunk
    def keyword_scores(self, q):
        return np.asarray(self.bm25.get_scores(tokenize(q)), dtype=np.float32)

    def semantic_scores(self, q):
        return self.emb @ embed_query(q, self.dims)  # vectors are unit-normalised: dot = cosine

    def _top(self, scores, k):
        order = np.argsort(-scores, kind="stable")[:k]
        return [{**self.chunks[i], "score": float(scores[i])} for i in order]

    # search modes: (question, k) -> ranked chunks
    def keyword(self, q, k=10):
        return self._top(self.keyword_scores(q), k)

    def semantic(self, q, k=10):
        return self._top(self.semantic_scores(q), k)

    def hybrid_weighted(self, q, k=10, alpha=0.7):
        def minmax(s):
            return (s - s.min()) / (s.max() - s.min() + 1e-9)
        scores = alpha * minmax(self.semantic_scores(q)) + (1 - alpha) * minmax(self.keyword_scores(q))
        return self._top(scores, k)

    def hybrid_rrf(self, q, k=10, rrf_k=60):
        fused = np.zeros(len(self.chunks))
        for s in (self.keyword_scores(q), self.semantic_scores(q)):
            ranks = np.empty(len(s))
            ranks[np.argsort(-s, kind="stable")] = np.arange(1, len(s) + 1)
            fused += 1.0 / (rrf_k + ranks)
        return self._top(fused, k)


indexes = {name: LocalIndex(chunks) for name, chunks in chunk_sets.items()}


def show(question: str, strategy: str, mode: str, k: int = 3):
    """Print the top-k chunks for one question. Use it to see why a strategy fails."""
    print(f"Q: {question}   [{strategy} / {mode}]")
    for h in getattr(indexes[strategy], mode)(question, k):
        snippet = h["text"][:140].replace("\n", " ")
        print(f"  {h['score']:7.3f}  {h['chunk_id']:<28} {snippet}")

### 4a. Keyword (BM25)

No AWS needed.

In [11]:
kw_results = {name: evaluate(idx.keyword) for name, idx in indexes.items()}
summarize(kw_results)

,hit@1,hit@3,mrr,ctx_tokens@3
whole_doc,0.944,1.000,0.972,1469.833
fixed_100w,0.722,0.889,0.824,408.778
fixed_50w,0.833,0.889,0.877,214.556
recursive_500c,0.722,0.944,0.829,275.167
markdown,0.833,1.000,0.898,271.278
markdown_ctx,0.778,1.000,0.870,300.611
qa_aware,0.944,1.000,0.972,150.833


In [12]:
# Lexical vs paraphrase questions: keyword search should do much worse on paraphrases
pd.concat(kw_results, names=["strategy"]).groupby(["strategy", "type"])["mrr"].mean().unstack().round(3)

type,lexical,paraphrase
strategy,,
fixed_100w,0.843,0.806
fixed_50w,0.843,0.911
markdown,1.000,0.796
markdown_ctx,0.944,0.796
qa_aware,0.944,1.000
recursive_500c,0.843,0.815
whole_doc,1.000,0.944


In [13]:
show("What should I spray on before using my flat iron?", "markdown_ctx", "keyword")
show("My scalp gets greasy really fast, what can help?", "markdown_ctx", "keyword")

Q: What should I spray on before using my flat iron?   [markdown_ctx / keyword]
    6.569  heat-shield-mist#2           Heat Shield Mist | Who it's for ## Who it's for Best for all hair types, including fine and color-treated hair. Anyone who blow-dries, flat-
    6.314  heat-shield-mist#4           Heat Shield Mist | How to use ## How to use Mist evenly over damp or dry hair before blow-drying, flat-ironing, or curling. Comb through to 
    3.288  repair-bond-oil#6            Repair Bond Oil | Common questions ## Common questions **Is the Repair Bond Oil safe for color-treated or bleached hair?** Yes. It is specif
Q: My scalp gets greasy really fast, what can help?   [markdown_ctx / keyword]
    5.912  scalp-reset-serum#6          Scalp Reset Serum | Common questions ## Common questions **Is the Scalp Reset Serum safe for color-treated hair?** Yes. It is applied to the
    3.313  faq#1                        AnyCompany Hair Care — General FAQ | Shipping ## Shipping **How much is shipp

### 4b. Semantic (Bedrock Titan Text Embeddings V2)

Titan V2 returns 256, 512 or 1024 dimensions and can return unit-normalised vectors, so a dot product equals cosine similarity. Embeddings are cached in `notebooks/.cache/` so reruns don't call Bedrock again.

**Cost:** embedding every strategy's chunks is roughly 30k tokens in total, well under one cent.

**If the next cell fails:**
- `NoCredentialsError` / `ExpiredToken`: refresh your AWS credentials (`aws sso login`, or set `AWS_PROFILE`) and restart the kernel.
- `AccessDeniedException`: your IAM identity needs `bedrock:InvokeModel` on the Titan V2 model, and the model has to be enabled in the Bedrock console for this region.
- `ValidationException` about the model ID: check that Titan Text Embeddings V2 is available in `REGION`.

In [14]:
import boto3

EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
EMBED_DIMS = 1024  # 256 | 512 | 1024

bedrock = boto3.client("bedrock-runtime", region_name=REGION)
_cache_path = CACHE_DIR / "embeddings.json"
_cache: dict[str, list[float]] = json.loads(_cache_path.read_text()) if _cache_path.exists() else {}


def embed(text: str, dims: int | None = None) -> list[float]:
    dims = dims or EMBED_DIMS
    key = hashlib.sha256(f"{EMBED_MODEL_ID}|{dims}|{text}".encode()).hexdigest()
    if key not in _cache:
        body = json.dumps({"inputText": text, "dimensions": dims, "normalize": True})
        resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body)
        _cache[key] = json.loads(resp["body"].read())["embedding"]
    return _cache[key]


def save_cache():
    _cache_path.write_text(json.dumps(_cache))


def embed_texts(texts: list[str], dims: int | None = None) -> np.ndarray:
    vecs = np.array([embed(t, dims) for t in texts], dtype=np.float32)
    save_cache()
    return vecs


def embed_query(q: str, dims: int | None = None) -> np.ndarray:
    return np.array(embed(q, dims), dtype=np.float32)


v = embed_query("smoke test")
print(f"dims={v.shape[0]}  norm={np.linalg.norm(v):.3f}")

dims=1024  norm=1.000


In [15]:
sem_results = {name: evaluate(idx.semantic) for name, idx in indexes.items()}
save_cache()
summarize(sem_results)

,hit@1,hit@3,mrr,ctx_tokens@3
whole_doc,0.833,0.944,0.898,1367.667
fixed_100w,0.889,1.000,0.944,412.944
fixed_50w,0.833,1.000,0.898,222.389
recursive_500c,0.944,1.000,0.963,268.389
markdown,1.000,1.000,1.000,248.500
markdown_ctx,0.889,0.944,0.917,276.778
qa_aware,0.944,0.944,0.944,153.889


In [16]:
show("What should I spray on before using my flat iron?", "markdown_ctx", "semantic")
show("Is niacinamide in any of your products?", "markdown_ctx", "semantic")

Q: What should I spray on before using my flat iron?   [markdown_ctx / semantic]
    0.326  heat-shield-mist#4           Heat Shield Mist | How to use ## How to use Mist evenly over damp or dry hair before blow-drying, flat-ironing, or curling. Comb through to 
    0.248  hair-type-guide#5            AnyCompany Hair-Type & Concern Guide | Fine hair ## Fine hair Goal: protection and definition without weight. - Protect: Heat Shield Mist (l
    0.245  air-dry-styling-gel#4        Air-Dry Styling Gel | How to use ## How to use Smooth over wet hair in sections, scrunch to encourage curl, and air-dry. Once fully dry, scr
Q: Is niacinamide in any of your products?   [markdown_ctx / semantic]
    0.265  faq#0                        AnyCompany Hair Care — General FAQ | About our products # AnyCompany Hair Care — General FAQ   ## About our products **Are AnyCompany produc
    0.226  silk-slip-conditioner#5      Silk Slip Conditioner | Ingredients & safety ## Ingredients & safety Color-safe and 

### 4c. Hybrid and the full comparison

Every strategy × every mode. The keyword and semantic numbers above come back as rows of this table.

In [17]:
MODES = ["keyword", "semantic", "hybrid_weighted", "hybrid_rrf"]

all_results = {
    (name, mode): evaluate(getattr(idx, mode))
    for name, idx in indexes.items()
    for mode in MODES
}
save_cache()

summary = summarize(all_results)
summary.index = pd.MultiIndex.from_tuples(summary.index, names=["strategy", "mode"])
summary["mrr"].unstack("mode")[MODES]

mode,keyword,semantic,hybrid_weighted,hybrid_rrf
strategy,,,,
fixed_100w,0.824,0.944,0.944,0.935
fixed_50w,0.877,0.898,0.958,0.903
markdown,0.898,1.000,0.972,0.972
markdown_ctx,0.870,0.917,0.972,0.944
qa_aware,0.972,0.944,0.958,0.931
recursive_500c,0.829,0.963,0.963,0.926
whole_doc,0.972,0.898,0.935,0.958


In [18]:
summary["hit@1"].unstack("mode")[MODES]

mode,keyword,semantic,hybrid_weighted,hybrid_rrf
strategy,,,,
fixed_100w,0.722,0.889,0.889,0.889
fixed_50w,0.833,0.833,0.944,0.833
markdown,0.833,1.000,0.944,0.944
markdown_ctx,0.778,0.889,0.944,0.889
qa_aware,0.944,0.944,0.944,0.889
recursive_500c,0.722,0.944,0.944,0.889
whole_doc,0.944,0.833,0.889,0.944


In [19]:
# Quality vs. how much text you hand the LLM
summary.sort_values(["mrr", "ctx_tokens@3"], ascending=[False, True]).head(12)

hit@1  hit@3    mrr  ctx_tokens@3
strategy       mode                                              
markdown       semantic         1.000  1.000  1.000       248.500
qa_aware       keyword          0.944  1.000  0.972       150.833
markdown       hybrid_rrf       0.944  1.000  0.972       254.333
               hybrid_weighted  0.944  1.000  0.972       254.944
markdown_ctx   hybrid_weighted  0.944  1.000  0.972       306.722
whole_doc      keyword          0.944  1.000  0.972      1469.833
recursive_500c semantic         0.944  1.000  0.963       268.389
               hybrid_weighted  0.944  1.000  0.963       282.722
qa_aware       hybrid_weighted  0.944  0.944  0.958       163.722
fixed_50w      hybrid_weighted  0.944  0.944  0.958       221.333
whole_doc      hybrid_rrf       0.944  0.944  0.958      1499.556
qa_aware       semantic         0.944  0.944  0.944       153.889

In [20]:
# Paraphrase vs lexical for each mode
per_q = pd.concat(all_results, names=["strategy", "mode"])
per_q.groupby(["strategy", "mode", "type"])["mrr"].mean().unstack("type").unstack("mode").round(2)

type              lexical                                  paraphrase  \
mode           hybrid_rrf hybrid_weighted keyword semantic hybrid_rrf   
strategy                                                                
fixed_100w           0.93            0.94    0.84     0.94       0.94   
fixed_50w            0.86            0.92    0.84     0.87       0.94   
markdown             1.00            1.00    1.00     1.00       0.94   
markdown_ctx         0.94            0.94    0.94     0.89       0.94   
qa_aware             0.92            0.92    0.94     0.89       0.94   
recursive_500c       0.85            0.93    0.84     1.00       1.00   
whole_doc            0.92            0.93    1.00     0.91       1.00   

type                                             
mode           hybrid_weighted keyword semantic  
strategy                                         
fixed_100w                0.94    0.81     0.94  
fixed_50w                 1.00    0.91     0.93  
markdown                  0.94    0.80     1.00  
markdown_ctx              1.00    0.80     0.94  
qa_aware                  1.00    1.00     1.00  
recursive_500c            1.00    0.81     0.93  
whole_doc                 0.94    0.94     0.89

### 4d. Tune the winner

Set `STRATEGY` to the best-looking row above, then:
1. sweep the hybrid weight `alpha` (0 = keyword only, 1 = semantic only)
2. check whether smaller embeddings (cheaper to store and search) lose anything
3. read the questions it still gets wrong

In [21]:
STRATEGY = "markdown_ctx"  # <- set from the tables above
idx = indexes[STRATEGY]

sweep = {
    alpha: evaluate(lambda q, k, a=alpha: idx.hybrid_weighted(q, k, alpha=a))
    for alpha in np.round(np.linspace(0, 1, 11), 1)
}
summarize(sweep).rename_axis("alpha")

,hit@1,hit@3,mrr,ctx_tokens@3
alpha,,,,
0.0,0.778,1.000,0.870,300.611
0.1,0.778,1.000,0.870,303.389
0.2,0.778,1.000,0.880,304.389
0.3,0.833,1.000,0.907,303.944
0.4,0.889,1.000,0.944,297.222
0.5,0.889,1.000,0.944,298.056
0.6,0.889,1.000,0.944,295.000
0.7,0.944,1.000,0.972,306.722
0.8,0.944,1.000,0.972,296.833


In [22]:
dim_results = {
    dims: evaluate(LocalIndex(chunk_sets[STRATEGY], dims=dims).semantic)
    for dims in (256, 512, 1024)
}
save_cache()
summarize(dim_results).rename_axis("dims")

,hit@1,hit@3,mrr,ctx_tokens@3
dims,,,,
256,0.778,0.889,0.824,281.000
512,0.889,0.889,0.908,277.778
1024,0.889,0.944,0.917,276.778


In [23]:
MODE = "hybrid_rrf"  # <- the mode you are leaning towards
misses = all_results[(STRATEGY, MODE)].query("`hit@1` == 0")[["q", "type", "mrr", "top3"]]
misses

,q,type,mrr,top3
3,Is niacinamide in any of your products?,lexical,0.5,"[faq, scalp-reset-serum, hair-type-guide]"
17,Will the product with argan oil make thin hair look oily?,paraphrase,0.5,"[hair-type-guide, repair-bond-oil, define-curl-cream]"


In [24]:
for q in misses.q:
    show(q, STRATEGY, MODE)
    print()

Q: Is niacinamide in any of your products?   [markdown_ctx / hybrid_rrf]
    0.033  faq#0                        AnyCompany Hair Care — General FAQ | About our products # AnyCompany Hair Care — General FAQ   ## About our products **Are AnyCompany produc
    0.030  scalp-reset-serum#3          Scalp Reset Serum | Key ingredients ## Key ingredients - Niacinamide — supports a balanced, comfortable scalp. - Tea Tree — clarifies and he
    0.030  hair-type-guide#0            # AnyCompany Hair-Type & Concern Guide   Use this guide to match customers to the right routine. All AnyCompany products are color-safe, veg

Q: Will the product with argan oil make thin hair look oily?   [markdown_ctx / hybrid_rrf]
    0.032  hair-type-guide#6            AnyCompany Hair-Type & Concern Guide | Oily or sensitive scalp ## Oily or sensitive scalp Goal: balance the scalp and control excess oil. - 
    0.032  repair-bond-oil#6            Repair Bond Oil | Common questions ## Common questions **Is the Repair 

When reading the misses, ask:
- Is the label wrong? Some questions are answered by more than one doc (e.g. the hair-type guide repeats product facts).
- Did the right doc come 2nd behind a near-duplicate? Then hit@3 matters more than hit@1, because the agent sees all top-k.
- Would a different chunker or a context prefix fix it, or does it need a reranker? Reranking is a v1 knob v2 doesn't have; Bedrock offers rerank models you could add later.

## Part 5. OpenSearch

### 5.1 Choose where to run it

| Option | Rough cost (us-east-1, check the pricing page) | Hybrid search | Use it for |
| --- | --- | --- | --- |
| **Local Docker** | free | yes (neural-search plugin is bundled) | **this notebook** |
| **OpenSearch Service domain**, 1 × `t3.small.search` | ~$0.036/h (~$26/month) + EBS | yes (2.11+) | v1 deployed by Terraform (`infra/modules/rag`). AWS recommends T3 for dev/QA only, which fits a demo. |
| **OpenSearch Serverless**, vector search collection | billed for a minimum number of OCUs even when idle, roughly $175/month+ | yes (min_max / l2 normalisation) | production with variable load; too expensive to leave running here |

**Recommendation:** develop against **local Docker** now. Deploy a **single-node managed domain** for v1 and destroy it when you're not measuring. The code below runs unchanged against both, apart from auth.

### 5.2 Start OpenSearch locally

Start Docker Desktop, then run this in a terminal. The security plugin is disabled, which is fine on localhost only:

```bash
docker run -d --name opensearch-rag \
  -p 9200:9200 -p 9600:9600 \
  -e "discovery.type=single-node" \
  -e "DISABLE_SECURITY_PLUGIN=true" \
  -e "DISABLE_INSTALL_DEMO_CONFIG=true" \
  -e "OPENSEARCH_JAVA_OPTS=-Xms1g -Xmx1g" \
  opensearchproject/opensearch:2.19.1

curl -s localhost:9200 | head -20   # wait ~20s for it to come up
```

Optional UI: run `opensearchproject/opensearch-dashboards:2.19.1` on port 5601 with `-e DISABLE_SECURITY_DASHBOARDS_PLUGIN=true -e OPENSEARCH_HOSTS='["http://host.docker.internal:9200"]'` to get the Dev Tools console.

Stop it with `docker stop opensearch-rag`, or `docker rm -f opensearch-rag` to delete it and its data.

### 5.3 Pointing at AWS instead (later)

Set `OPENSEARCH_TARGET=aws` and `OPENSEARCH_HOST=<domain endpoint without https://>` (from `terraform output` once the rag module exists). The client signs requests with your AWS credentials (SigV4, service `es`, or `aoss` for Serverless). Your IAM identity must be allowed by the domain's access policy.

In [25]:
from opensearchpy import AWSV4SignerAuth, OpenSearch, RequestsHttpConnection, helpers

OPENSEARCH_TARGET = os.getenv("OPENSEARCH_TARGET", "local")  # "local" | "aws"

if OPENSEARCH_TARGET == "local":
    os_client = OpenSearch(hosts=[{"host": "localhost", "port": 9200}], use_ssl=False, timeout=30)
else:
    service = os.getenv("OPENSEARCH_SERVICE", "es")  # "es" = managed domain, "aoss" = Serverless
    auth = AWSV4SignerAuth(boto3.Session().get_credentials(), REGION, service)
    os_client = OpenSearch(
        hosts=[{"host": os.environ["OPENSEARCH_HOST"], "port": 443}],
        http_auth=auth, use_ssl=True, verify_certs=True,
        connection_class=RequestsHttpConnection, timeout=30,
    )

info = os_client.info()
print(info["version"]["distribution"], info["version"]["number"])

opensearch 2.19.1


### 5.4 Create the index

The mapping is a first draft of what the v1 ingestion pipeline will create:

- `text` uses the `english` analyzer (stemming: *irons → iron*, *colored → color*). Compare it with your local BM25, which has no stemming.
- `title` and `section` are separate fields, so you can boost them in keyword queries.
- `embedding` is a `knn_vector` using HNSW on the Lucene engine with cosine similarity. Lucene suits small indexes, needs no native-library memory tuning, and supports filters during k-NN search.

In [26]:
INDEX = f"kb-{STRATEGY.replace('_', '-')}"
chunks = chunk_sets[STRATEGY]
vectors = indexes[STRATEGY].emb  # reuse the cached embeddings from Part 4

index_body = {
    "settings": {"index": {"knn": True, "number_of_shards": 1, "number_of_replicas": 0}},
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "doc_id": {"type": "keyword"},
            "title": {"type": "text", "analyzer": "english"},
            "section": {"type": "text", "analyzer": "english"},
            "text": {"type": "text", "analyzer": "english"},
            "embedding": {
                "type": "knn_vector",
                "dimension": int(vectors.shape[1]),
                "method": {"name": "hnsw", "engine": "lucene", "space_type": "cosinesimil"},
            },
        }
    },
}
# Serverless doesn't accept shard/replica settings
if OPENSEARCH_TARGET == "aws" and os.getenv("OPENSEARCH_SERVICE") == "aoss":
    index_body["settings"]["index"] = {"knn": True}

if os_client.indices.exists(index=INDEX):
    os_client.indices.delete(index=INDEX)
os_client.indices.create(index=INDEX, body=index_body)

actions = (
    {"_index": INDEX, "_id": c["chunk_id"], "_source": {**c, "embedding": v.tolist()}}
    for c, v in zip(chunks, vectors)
)
ok, errors = helpers.bulk(os_client, actions, refresh=True)
print(f"indexed {ok} chunks into {INDEX!r}, errors: {errors}")

indexed 76 chunks into 'kb-markdown-ctx', errors: []


### 5.5 The three query types

In [27]:
SOURCE = {"excludes": ["embedding"]}


def os_hits(resp) -> list[dict]:
    return [{**h["_source"], "score": h["_score"]} for h in resp["hits"]["hits"]]


def keyword_query(q: str) -> dict:
    return {"multi_match": {"query": q, "fields": ["text", "title^2", "section"]}}


def knn_query(q: str, k: int) -> dict:
    return {"knn": {"embedding": {"vector": embed_query(q).tolist(), "k": k}}}


def os_keyword(q, k=10):
    return os_hits(os_client.search(index=INDEX, body={"size": k, "query": keyword_query(q), "_source": SOURCE}))


def os_semantic(q, k=10):
    return os_hits(os_client.search(index=INDEX, body={"size": k, "query": knn_query(q, k), "_source": SOURCE}))


os_keyword("flat iron spray")[:3]

[{'section': "Who it's for",
  'text': "Heat Shield Mist | Who it's for\n## Who it's for\nBest for all hair types, including fine and color-treated hair. Anyone who blow-dries, flat-irons, or curls should use it.",
  'title': 'Heat Shield Mist',
  'doc_id': 'heat-shield-mist',
  'chunk_id': 'heat-shield-mist#2',
  'score': 7.5550604},
 {'section': 'How to use',
  'text': 'Heat Shield Mist | How to use\n## How to use\nMist evenly over damp or dry hair before blow-drying, flat-ironing, or curling. Comb through to distribute, then style as usual.',
  'title': 'Heat Shield Mist',
  'doc_id': 'heat-shield-mist',
  'chunk_id': 'heat-shield-mist#4',
  'score': 7.4529457},
 {'section': 'Shipping',
  'text': 'AnyCompany Hair Care — General FAQ | Shipping\n## Shipping\n**How much is shipping?**\nStandard shipping is free on orders over $45. Orders under $45 have a flat $5 shipping fee.  \n**How fast will my order arrive?**\nStandard shipping is 3–5 business days. Expedited 2-day shipping is avai

**Hybrid** needs a *search pipeline*. The `hybrid` query runs both sub-queries, and the pipeline's `normalization-processor` min-max normalises each score list and combines them with weights (in the same order as the sub-queries, so here `[keyword, semantic]`). This matches `LocalIndex.hybrid_weighted`, so set the weights from your alpha sweep.

In [28]:
ALPHA = 0.7  # semantic weight, from the 4d sweep
PIPELINE = "kb-hybrid"

os_client.transport.perform_request("PUT", f"/_search/pipeline/{PIPELINE}", body={
    "description": "min-max normalise BM25 and k-NN scores, then weighted mean",
    "phase_results_processors": [{
        "normalization-processor": {
            "normalization": {"technique": "min_max"},
            "combination": {"technique": "arithmetic_mean", "parameters": {"weights": [1 - ALPHA, ALPHA]}},
        }
    }],
})


def os_hybrid(q, k=10):
    body = {
        "size": k,
        "_source": SOURCE,
        "query": {"hybrid": {"queries": [keyword_query(q), knn_query(q, max(k, 20))]}},
    }
    return os_hits(os_client.search(index=INDEX, body=body, params={"search_pipeline": PIPELINE}))


for h in os_hybrid("What should I spray on before using my flat iron?", 3):
    print(f"{h['score']:.3f}  {h['chunk_id']}")

0.994  heat-shield-mist#4
0.300  heat-shield-mist#2
0.027  hair-type-guide#5


### 5.6 Does OpenSearch reproduce the local results?

Expect them to be close but not identical: OpenSearch uses the `english` analyzer and field boosts, HNSW is approximate, and BM25 parameters differ slightly. A big gap usually means a mapping or query bug.

In [29]:
os_results = {
    ("opensearch", "keyword"): evaluate(os_keyword),
    ("opensearch", "semantic"): evaluate(os_semantic),
    ("opensearch", "hybrid_weighted"): evaluate(os_hybrid),
    ("local", "keyword"): all_results[(STRATEGY, "keyword")],
    ("local", "semantic"): all_results[(STRATEGY, "semantic")],
    ("local", "hybrid_weighted"): evaluate(lambda q, k: idx.hybrid_weighted(q, k, alpha=ALPHA)),
}
summarize(os_results)

hit@1  hit@3    mrr  ctx_tokens@3
opensearch keyword          0.833  0.944  0.889       323.444
           semantic         0.889  0.944  0.917       273.111
           hybrid_weighted  0.944  0.944  0.950       303.556
local      keyword          0.778  1.000  0.870       300.611
           semantic         0.889  0.944  0.917       276.778
           hybrid_weighted  0.944  1.000  0.972       306.722

In [30]:
import time

def latency_ms(fn, n=20, q="Is the bond oil safe for bleached hair?"):
    fn(q)  # warm-up (also caches the query embedding)
    t = []
    for _ in range(n):
        start = time.perf_counter(); fn(q); t.append((time.perf_counter() - start) * 1000)
    return {"p50_ms": round(np.percentile(t, 50), 1), "p95_ms": round(np.percentile(t, 95), 1)}

# Query embeddings are cached, so this times OpenSearch only. Bedrock embedding adds tens of ms per real query.
pd.DataFrame({name: latency_ms(fn) for name, fn in
              {"keyword": os_keyword, "semantic": os_semantic, "hybrid": os_hybrid}.items()}).T

,p50_ms,p95_ms
keyword,12.7,19.9
semantic,9.8,17.3
hybrid,14.1,19.3


### 5.7 Clean up

```python
os_client.indices.delete(index=INDEX)
os_client.transport.perform_request("DELETE", f"/_search/pipeline/{PIPELINE}")
```
Then run `docker stop opensearch-rag`. If you pointed this at an AWS domain, destroy it when you're done measuring.

## Part 6. Decisions → RAG service brief

Fill this in, then copy it into the RAG service brief. These are the v1 choices to compare against v2 on Day 3–4.

| Decision | Choice | Evidence (cell / metric) |
| --- | --- | --- |
| Chunking strategy & size | | |
| Contextual header in chunks? | | |
| Embedding model & dims | `amazon.titan-embed-text-v2:0`, ? | |
| Retrieval mode | | |
| Hybrid weights / RRF | | |
| top-k returned to the agent | | |
| Keyword analyzer | `english`? | |
| OpenSearch deployment | managed domain, `t3.small.search`, 1 node? | |

**How v1 will differ from v2 (start the Day 3 list here):** chunking (custom vs. KB fixed/semantic/hierarchical), hybrid weighting control, analyzer choice, index-time context headers.

**Questions to carry into the golden set:** the misses from 4d, and any question whose label you weren't sure about.